# Notebook 05 — Network Centrality Measures

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 05 of 12  
**Time estimate:** 75 minutes

> Degree centrality asks 'who is most popular?'
> Betweenness centrality asks 'who is the gatekeeper?'
> In biological networks, these often disagree — and the disagreement is biologically informative.

---
## Step 1 — Motivation

Which protein in a PPI network is most 'important'? The answer depends on what
'important' means. A protein can be important because it has many partners (hub),
because all shortest paths flow through it (bottleneck), because it is close to
everyone (closeness), or because it has influential neighbors (eigenvector). Each
centrality captures a different aspect of network position.

---
## Step 2 — Intuition

Think of a PPI network as a city road network:
- **Degree:** how many roads connect to this intersection
- **Betweenness:** how many people's routes pass through this intersection (bottleneck traffic)
- **Closeness:** how close is this intersection to all others (emergency response speed)
- **Eigenvector:** being connected to well-connected intersections (prestige)
- **PageRank:** like eigenvector but with a random-walk damping term (robust to dead ends)

A protein that is a **hub** (high degree) but NOT a **bottleneck** (low betweenness)
is embedded in a dense cluster — its loss is compensated by cluster partners.
A protein that is a **bottleneck** but NOT a hub (low degree, high betweenness) is
a "bridge" protein connecting otherwise separate modules — its loss is catastrophic.

---
## Step 3 — Biological Background

**Hub-bottleneck analysis (Yu et al. 2007):**
- **Date hubs:** interact with different partners at different times (high betweenness)
- **Party hubs:** interact with all partners simultaneously (lower betweenness)
- Date hubs are more likely to be essential — they bridge modules

**Drug targets:** Many approved drugs target hub proteins.
- Kinases (high degree in signaling networks) are the largest drug target class
- Betweenness-central proteins are attractive targets because blocking them disrupts
  many pathways simultaneously

**Transcription factors:** Often low-degree in PPI but high-betweenness in GRNs —
they regulate many downstream genes through few physical interactions.

**PageRank in biology:** Applied to metabolic networks to rank metabolites by their
centrality in metabolic flux; also to gene networks for prioritizing disease genes.

---
## Step 4 — Mathematical Explanation

Let $G = (V, E)$ with $n = |V|$, $m = |E|$. All paths are shortest paths unless stated.

**Degree centrality:**
$$C_D(v) = \frac{k_v}{n-1}$$

**Betweenness centrality (Brandes 2001):**
$$C_B(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$$
where $\sigma_{st}$ = number of shortest paths from $s$ to $t$, $\sigma_{st}(v)$ = those through $v$.

Brandes algorithm: BFS forward pass accumulates $\sigma$ and predecessor lists;
backward pass accumulates partial dependencies. Complexity: $O(nm)$ unweighted.

**Closeness centrality:**
$$C_C(v) = \frac{n-1}{\sum_{u \neq v} d(u,v)}$$
Inverse average distance to all other nodes.

**Eigenvector centrality:**
$$C_E(v) = \frac{1}{\lambda} \sum_{u \in N(v)} C_E(u) \quad \Rightarrow \quad A\mathbf{x} = \lambda \mathbf{x}$$
The centrality vector $\mathbf{x}$ is the leading eigenvector of the adjacency matrix.

**PageRank:**
$$PR(v) = \frac{1-d}{n} + d \sum_{u \to v} \frac{PR(u)}{k_u^{out}}$$
where $d \approx 0.85$ is the damping factor. Power iteration converges to the stationary
distribution of a random walk with teleportation.

In [ ]:
# Step 6 — Python: Centrality measures — implement key ones from scratch, NetworkX for betweenness
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

# ---- Re-use BA network from NB04 ----
def barabasi_albert(n_nodes, m, seed=42):
    rng = np.random.default_rng(seed)
    seed_n = m + 1
    adj = {i: set() for i in range(seed_n)}
    edges = []
    for i in range(seed_n):
        for j in range(i+1, seed_n):
            adj[i].add(j); adj[j].add(i)
            edges.append((i, j))
    stub_list = []
    for u, v in edges:
        stub_list += [u, v]
    for new_node in range(seed_n, n_nodes):
        adj[new_node] = set()
        targets = set()
        while len(targets) < m:
            c = rng.choice(stub_list)
            if c != new_node: targets.add(c)
        for t in targets:
            adj[new_node].add(t); adj[t].add(new_node)
            edges.append((new_node, t))
            stub_list += [new_node, t]
    return adj, edges

adj, edges = barabasi_albert(200, m=2)
nodes = sorted(adj.keys())
n = len(nodes)

# ---- 1. Degree centrality (from scratch) ----
deg_cent = {v: len(adj[v]) / (n - 1) for v in nodes}

# ---- 2. Closeness centrality (from scratch via BFS) ----
def bfs_distances(adj, source):
    dist = {source: 0}
    q = deque([source])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

def closeness_centrality(adj, nodes):
    cc = {}
    for v in nodes:
        dists = bfs_distances(adj, v)
        reachable = len(dists) - 1
        if reachable == 0:
            cc[v] = 0.0
        else:
            cc[v] = reachable / sum(dists.values())
    return cc

close_cent = closeness_centrality(adj, nodes)
print('Top 5 by closeness:', sorted(close_cent, key=close_cent.get, reverse=True)[:5])

# ---- 3. Eigenvector centrality (from scratch via power iteration) ----
def eigenvector_centrality(adj, nodes, n_iter=100, tol=1e-6):
    n = len(nodes)
    idx = {v: i for i, v in enumerate(nodes)}
    x = np.ones(n) / n
    for _ in range(n_iter):
        x_new = np.zeros(n)
        for v in nodes:
            for u in adj[v]:
                x_new[idx[v]] += x[idx[u]]
        norm = np.linalg.norm(x_new)
        if norm > 0: x_new /= norm
        if np.linalg.norm(x_new - x) < tol:
            break
        x = x_new
    return {v: x[idx[v]] for v in nodes}

eig_cent = eigenvector_centrality(adj, nodes)

# ---- 4. Betweenness centrality (NetworkX — O(nm) Brandes algorithm) ----
import networkx as nx
G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)
btwn_cent = nx.betweenness_centrality(G, normalized=True)
pagerank = nx.pagerank(G, alpha=0.85)

# Collect all centralities into arrays for comparison
deg_arr = np.array([deg_cent[v] for v in nodes])
close_arr = np.array([close_cent[v] for v in nodes])
eig_arr = np.array([eig_cent[v] for v in nodes])
btwn_arr = np.array([btwn_cent[v] for v in nodes])
pr_arr = np.array([pagerank[v] for v in nodes])

print('\nCorrelation matrix: degree / closeness / eigenvector / betweenness / PageRank')
cent_matrix = np.column_stack([deg_arr, close_arr, eig_arr, btwn_arr, pr_arr])
corr = np.corrcoef(cent_matrix.T).round(3)
print(corr)
print('\nNote: degree and eigenvector are highly correlated; betweenness is less correlated')
print('→ Betweenness identifies different nodes than pure degree')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Use circular layout
theta = np.linspace(0, 2*np.pi, n, endpoint=False)
pos = np.column_stack([np.cos(theta), np.sin(theta)])
node_idx = {v: i for i, v in enumerate(nodes)}

def draw_network(ax, adj, pos, node_idx, color_arr, size_arr, title, cmap='Reds'):
    for u in adj:
        for v in adj[u]:
            if u < v:
                ui, vi = node_idx[u], node_idx[v]
                ax.plot([pos[ui,0], pos[vi,0]], [pos[ui,1], pos[vi,1]],
                          'grey', lw=0.2, alpha=0.3, zorder=1)
    sc = ax.scatter(pos[:,0], pos[:,1], c=color_arr, s=size_arr,
                      cmap=cmap, zorder=2, alpha=0.8)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
    return sc

sc1 = draw_network(axes[0], adj, pos, node_idx,
                    deg_arr, 20 + 200*deg_arr/deg_arr.max(),
                    'A. Degree centrality\n(size ∝ degree, color = degree centrality)')
plt.colorbar(sc1, ax=axes[0], shrink=0.7, label='C_D')

sc2 = draw_network(axes[1], adj, pos, node_idx,
                    btwn_arr, 20 + 200*btwn_arr/max(btwn_arr.max(), 1e-10),
                    'B. Betweenness centrality\n(size ∝ betweenness, color = C_B)')
plt.colorbar(sc2, ax=axes[1], shrink=0.7, label='C_B')

# Panel C: Hub vs. bottleneck scatter
ax = axes[2]
ax.scatter(deg_arr, btwn_arr, c=pr_arr, cmap='viridis', s=30, alpha=0.7)
# Annotate top bottlenecks (high betweenness, moderate degree)
top_btwn_idx = np.argsort(btwn_arr)[-8:]
for i in top_btwn_idx:
    ax.annotate(str(nodes[i]), (deg_arr[i], btwn_arr[i]),
                  fontsize=6, ha='left', va='bottom', alpha=0.7)
ax.set_xlabel('Degree centrality')
ax.set_ylabel('Betweenness centrality')
ax.set_title('C. Hub vs. Bottleneck\n(color = PageRank)')

plt.suptitle('Module 12 NB05: Network Centrality Measures', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('network_centrality.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Identify the top-5 nodes by betweenness but NOT in the top-10 by degree.
   These are the "bottlenecks." In a real PPI network, what biological role might
   such proteins play?
2. Implement PageRank from scratch using power iteration. Verify it matches
   `nx.pagerank()` to within 0.001 per node.
3. What happens to betweenness centrality if you add a second shortest path between
   two nodes that previously had only one? (*Hint: revisit the definition.*)
4. Compute eigenvector centrality using `np.linalg.eig(A)` directly (not power
   iteration). What is the relationship between the leading eigenvalue and the
   network's "spectral radius"?

---
## Step 10 — Quiz

1. What does betweenness centrality measure? How is it computed?
2. What is the difference between a hub and a bottleneck?
3. Why does eigenvector centrality give high scores to proteins connected to
   well-connected proteins?
4. What is the damping factor in PageRank, and what does it represent?
5. Which centrality measure best predicts essential genes in yeast PPI networks?

---
## Step 12 — Reflection

> *[In one paragraph: explain why a protein with low degree but high betweenness
> centrality is a potential drug target, using a concrete biological scenario.]*

---
*Next: `06_community_detection_in_networks.ipynb`*